In [1]:
pwd

'C:\\Users\\hamza\\Programs\\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor\\detection\\fintuning'

In [7]:
images_dir, data_cfg

(WindowsPath('images/val'),
 {'path': 'C:\\Users\\hamza\\Datasets\\TrafficDatasets\\UA_DETRAC_robotflow_yolo',
  'train': 'images/train',
  'val': 'images/val',
  'nc': 5,
  'names': {0: 'vehicles', 1: 'track', 2: 'car', 3: 'van', 4: 'bus'}})

## Predictions:

In [10]:
from ultralytics import YOLO
from pathlib import Path
import yaml

# -----------------------
# Paths
# -----------------------
model_path = "yolov8n.pt"  # or your fine-tuned .pt
root = r"C:\Users\hamza\Datasets\TrafficDatasets\UA_DETRAC_robotflow_ultralytics"
data_yaml = root + r"\data.yaml"
save_dir = Path("predictions")
save_dir.mkdir(parents=True, exist_ok=True)

# -----------------------
# Load dataset yaml
# -----------------------
with open(data_yaml, "r") as f:
    data_cfg = yaml.safe_load(f)

# Use train / val / test as you want
images_dir = Path(root + "/"+ data_cfg["val"])  ## or "train" or "test"

# -----------------------
# Load model
# -----------------------
model = YOLO(model_path)

# -----------------------
# Run inference ONLY
# -----------------------
results = model.predict(
    source=str(images_dir),
    conf=0.001,          # very low to keep all predictions
    iou=1.0,             # disable suppression effects as much as possible
    save=False,
    save_txt=True,       # saves raw predictions per image
    save_conf=True,      # include confidence scores
    project=save_dir,
    name="raw_preds",
    verbose=False
)

print(f"Predictions saved to: {save_dir / 'raw_preds'}")


Results saved to C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor\detection\fintuning\predictions\raw_preds
500 labels saved to C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor\detection\fintuning\predictions\raw_preds\labels
Predictions saved to: predictions\raw_preds


In [12]:
import numpy as np
import pandas as pd

def compute_pr_ap(tp, conf, n_gt):
    tp = np.array(tp)
    conf = np.array(conf)

    # Sort by confidence (descending)
    order = np.argsort(-conf)
    tp = tp[order]
    conf = conf[order]

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(1 - tp)

    precision = tp_cum / (tp_cum + fp_cum + 1e-9)
    recall = tp_cum / n_gt

    # Interpolated AP
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    ap = np.sum((mrec[1:] - mrec[:-1]) * mpre[1:])

    table = pd.DataFrame({
        "confidence": conf,
        "TP_cum": tp_cum,
        "FP_cum": fp_cum,
        "precision": precision,
        "recall": recall
    })

    return table, ap


# ---------------- Example usage ---------------- #

# Example predictions (4 detections)
tp   = [1, 0, 1, 0]              # TP / FP
conf = [0.92, 0.85, 0.60, 0.30]  # confidence scores
n_gt = 7                         # total GT objects

table, ap = compute_pr_ap(tp, conf, n_gt)

print(table)
print("\nAP =", round(ap, 4))

   confidence  TP_cum  FP_cum  precision    recall
0        0.92       1       0   1.000000  0.142857
1        0.85       1       1   0.500000  0.142857
2        0.60       2       1   0.666667  0.285714
3        0.30       2       2   0.500000  0.285714

AP = 0.2381


In [ ]:
# !python coco_to_yolo.py
# !python restructure_yolo_dataset.py
# !python dawn_to_ultralytics.py

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")  
model.info()

In [ ]:
model = YOLO("yolo26n_finetuned.pt")  
model.info()

In [ ]:
model = YOLO("yolov8n.pt")  
model.info()

In [ ]:
model = YOLO("yolov8n_finetuned.pt")  
model.info()

In [ ]:
model = YOLO("yolo11n.pt")  
model.info()

In [ ]:
model = YOLO("yolo11n_finetuned.pt")  
model.info()

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")                    # rtdetr-l.pt
model.export(format="onnx")

In [ ]:
SEQUENCES_DIR = Path(
    r"C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor"
    r"\counting\experiments\IMAROC2\sequences"
)

OUTPUT_JSON = Path(
    r"C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor"
    r"\counting\experiments\IMAROC2\yolov8n_finetuned_imaroc2.json"
)


In [1]:
pip install ijson

Note: you may need to restart the kernel to use updated packages.


In [10]:

#!/usr/bin/env python3
"""
Split a very large JSON file of the form:

{
  "meta": {...},
  "sequences": {
      "kech36": {...},
      "kech37": {...},
      ...
  }
}

into:
  output/
    meta.json
    sequences/
      kech36.json
      kech37.json
      ...

✔ Streaming (low RAM)
✔ Handles Decimal safely
✔ Atomic writes
"""

import json
from pathlib import Path
from decimal import Decimal
import ijson

# -------------------------------------------------
# PATHS (ADJUST THESE)
# -------------------------------------------------
BIG_JSON = Path(
    r"C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor"
    r"\counting\experiments\IMAROC2\yolo26n_finetuned_imaroc2.json"
)

OUT_ROOT = Path(
    r"C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor"
    r"\counting\experiments\IMAROC2\sequences_yolo26n_finetuned"
)

SEQUENCES_DIR = OUT_ROOT / "sequences"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
SEQUENCES_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Decimal-safe JSON serializer
# -------------------------------------------------
def json_default(obj):
    if isinstance(obj, Decimal):
        # choose ONE:
        return float(obj)   # OR: return str(obj) for exact precision
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

# -------------------------------------------------
# 1) Extract META (streaming)
# -------------------------------------------------
print("[INFO] Extracting meta.json ...")

with BIG_JSON.open("rb") as fh:
    try:
        meta = next(ijson.items(fh, "meta"))
    except StopIteration:
        raise RuntimeError("Top-level 'meta' key not found in big JSON")

tmp_meta = OUT_ROOT / "meta.tmp.json"
with tmp_meta.open("w", encoding="utf-8") as fh:
    json.dump(meta, fh, ensure_ascii=False, indent=2, default=json_default)
tmp_meta.replace(OUT_ROOT / "meta.json")

print("[OK] meta.json written")

# -------------------------------------------------
# 2) Extract SEQUENCES (one by one, streaming)
# -------------------------------------------------
print("[INFO] Extracting sequences ...")

count = 0
failed = 0

with BIG_JSON.open("rb") as fh:
    for seq_key, seq_obj in ijson.kvitems(fh, "sequences"):
        try:
            out_path = SEQUENCES_DIR / f"{seq_key}.json"
            tmp_path = out_path.with_suffix(".tmp.json")

            with tmp_path.open("w", encoding="utf-8") as ofh:
                json.dump(
                    seq_obj,
                    ofh,
                    ensure_ascii=False,
                    indent=2,
                    default=json_default
                )

            tmp_path.replace(out_path)
            count += 1

            if count % 50 == 0:
                print(f"[INFO] {count} sequences extracted...")

        except Exception as e:
            failed += 1
            print(f"[WARN] Failed to write sequence '{seq_key}': {e}")

print(f"[DONE] Extracted {count} sequences ({failed} failed)")
print(f"[OK] Output folder: {OUT_ROOT}")

[INFO] Extracting meta.json ...
[OK] meta.json written
[INFO] Extracting sequences ...
[INFO] 50 sequences extracted...
[DONE] Extracted 70 sequences (0 failed)
[OK] Output folder: C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor\counting\experiments\IMAROC2\sequences_yolo26n_finetuned


In [ ]:
import json
from pathlib import Path

# -----------------------
# PATHS (adjust if needed)
# -----------------------
SEQUENCES_DIR = Path(
    r"C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor"
    r"\counting\experiments\IMAROC2\sequences"
)

BIG_JSON_PATH = Path(
    r"C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor"
    r"\counting\experiments\IMAROC2\rtdetrl_imaroc2.json"
)

# -----------------------
# Load big JSON
# -----------------------
if BIG_JSON_PATH.exists():
    with BIG_JSON_PATH.open("r", encoding="utf-8") as f:
        big_data = json.load(f)
else:
    raise FileNotFoundError(f"Big JSON not found: {BIG_JSON_PATH}")

big_data.setdefault("sequences", {})

# -----------------------
# Merge sequences
# -----------------------
added = 0
skipped = 0

for seq_file in sorted(SEQUENCES_DIR.glob("*.json")):
    seq_key = seq_file.stem  # e.g. kech36

    if seq_key in big_data["sequences"]:
        skipped += 1
        continue

    with seq_file.open("r", encoding="utf-8") as f:
        seq_obj = json.load(f)

    big_data["sequences"][seq_key] = seq_obj
    added += 1

# -----------------------
# Write back big JSON (atomic)
# -----------------------
tmp_path = BIG_JSON_PATH.with_suffix(".tmp.json")
with tmp_path.open("w", encoding="utf-8") as f:
    json.dump(big_data, f, ensure_ascii=False, indent=2)

tmp_path.replace(BIG_JSON_PATH)

print(f"[DONE] Added {added} sequences, skipped {skipped}.")
print(f"[OK] Big JSON updated: {BIG_JSON_PATH}")

In [ ]:
#!/usr/bin/env python3
"""
Combine per-sequence JSON files under a `sequences` folder plus the experiments `meta.json`
into one single big JSON file (atomic write).

Adjust the two PATH constants below if needed.
"""
import json
from pathlib import Path
from datetime import datetime

# -----------------------
# PATHS (adjust if needed)
# -----------------------
SEQUENCES_DIR = Path(
    r"C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor"
    r"\counting\experiments\IMAROC2\sequences"
)

OUTPUT_JSON = Path(
    r"C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Based_SMART_sensor"
    r"\counting\experiments\IMAROC2\yolov8n_finetuned_imaroc2.json"
)

# -----------------------
# Basic checks
# -----------------------
if not SEQUENCES_DIR.exists() or not SEQUENCES_DIR.is_dir():
    raise FileNotFoundError(f"Sequences directory not found: {SEQUENCES_DIR}")

EXPERIMENTS_DIR = SEQUENCES_DIR.parent
META_PATH = EXPERIMENTS_DIR / "meta.json"

# -----------------------
# Load or create meta
# -----------------------
if META_PATH.exists():
    try:
        with META_PATH.open("r", encoding="utf-8") as fh:
            meta = json.load(fh)
    except Exception as e:
        print(f"[WARN] Failed to load meta.json ({META_PATH}): {e}; using fallback meta.")
        meta = {"timestamp_utc": datetime.utcnow().isoformat() + "Z", "note": "meta.json corrupted; fallback used"}
else:
    meta = {"timestamp_utc": datetime.utcnow().isoformat() + "Z", "note": "meta.json not found; autogenerated"}

# -----------------------
# Merge sequence files
# -----------------------
big_data = {"meta": meta, "sequences": {}}
added = 0
skipped = 0
failed = 0

for seq_file in sorted(SEQUENCES_DIR.glob("*.json")):
    # defensively skip files that look like meta / temp / partial
    if seq_file.name.lower() in ("meta.json", "meta.tmp.json"):
        skipped += 1
        continue

    seq_key = seq_file.stem  # e.g. kech36
    try:
        with seq_file.open("r", encoding="utf-8") as fh:
            seq_obj = json.load(fh)
    except Exception as e:
        print(f"[WARN] Failed to read/parse {seq_file}: {e}")
        failed += 1
        continue

    if seq_key in big_data["sequences"]:
        skipped += 1
        continue

    big_data["sequences"][seq_key] = seq_obj
    added += 1

# -----------------------
# Atomic write out
# -----------------------
tmp_path = OUTPUT_JSON.with_suffix(".tmp.json")
with tmp_path.open("w", encoding="utf-8") as fh:
    json.dump(big_data, fh, ensure_ascii=False, indent=2)
tmp_path.replace(OUTPUT_JSON)

print(f"[DONE] Added {added} sequences, skipped {skipped}, failed {failed}.")
print(f"[OK] Big JSON created: {OUTPUT_JSON}")


In [ ]:
from yolo_visualizer import YOLOVisualizer
from pathlib import Path

dataset_root = Path(r"C:\Users\hamza\Datasets\TrafficDatasets\traffic_detection_project_ultralytics")
dataset_root = Path(r"C:\Users\hamza\Datasets\TrafficDatasets\UA_DETRAC_robotflow_ultralytics")
dataset_root = Path(r"C:\Users\hamza\Datasets\TrafficDatasets\dawn_ultralytics")

visualizer = YOLOVisualizer(dataset_root)

visualizer.visualize_by_index(index = 532)
# visualizer.visualize_by_filename(filename = "td48_178_jpg.rf.859d37c56150c92f36d2dfac67a4f253.jpg", split="train")

In [ ]:
from yolo_dataset_analysis import YOLODatasetStats
dataset_root=r"C:\Users\hamza\Datasets\TrafficDatasets\traffic_detection_project_ultralytics"
dataset_root = Path(r"C:\Users\hamza\Datasets\TrafficDatasets\UA_DETRAC_robotflow_ultralytics")
dataset_root = Path(r"C:\Users\hamza\Datasets\TrafficDatasets\dawn_ultralytics")

stats = YOLODatasetStats(dataset_root)

In [ ]:
# 1️⃣ Print class frequencies
print(stats.class_frequencies())          # all splits
print(stats.class_frequencies("train"))   # train only
print(stats.class_frequencies("val"))   # train only

# 2️⃣ Plot bar chart
stats.plot_class_distribution()           # all
stats.plot_class_distribution("val")      # validation only

In [ ]:
# from annotation_standarizer import YOLODatasetStandardizer
# standard_names = [
#     "truck",
#     "car",
#     "van",
#     "bus"
# ]

# standardizer = YOLODatasetStandardizer(
#     standardized_root=r"C:\Users\hamza\Datasets\TrafficDatasets\standarized_dataset",
#     standard_names=standard_names
# )

# uadetrac_map = {
#     0: None,  # vehicles (removed)
#     1: 0,     # truck → four_wheels
#     2: 1,     # car → four_wheels
#     3: 2,     # van → four_wheels
#     4: 3      # bus → four_wheels
# }

# standardizer.add_dataset(
#     dataset_root=r"C:\Users\hamza\Datasets\TrafficDatasets\UA_DETRAC_robotflow_ultralytics",
#     class_id_map=uadetrac_map
# )


In [ ]:
from ultralytics import YOLO

yaml_path = r"C:\Users\hamza\Datasets\TrafficDatasets\standarized_dataset\data.yaml"
model = YOLO("yolo26n.pt")
# model.train(data=yaml_path, epochs=50, batch=8)

In [ ]:
# from ultralytics import YOLO
# model = YOLO("yolov8n.pt")
# results = model.val(
#     data=yaml_path,
#     imgsz=640,
#     batch=16,
#     device=0
# )